In [1]:
import pandas as pd


In [2]:
df = pd.read_csv("olist_orders_dataset.csv")

# 1) Exploratory Data Analysis
## 1.1) DATA Insception 

In [3]:
df.head(5)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


## 1.2) Statistical Summary

In [4]:
df.describe()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
count,99441,99441,99441,99441,99281,97658,96476,99441
unique,99441,99441,8,98875,90733,81018,95664,459
top,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2018-03-31 15:08:21,2018-02-27 04:31:10,2018-05-09 15:48:00,2018-05-14 20:02:44,2017-12-20 00:00:00
freq,1,1,96478,3,9,47,3,522


<span style="font-size: 20px;">Since the customer_id has all rows unique, we can assume there are no duplicate rows. 
But further, let's check if there are any duplicate rows using the **duplicated()** function.</span>

#### TO CHECK IF THERE ARE ANY DUPLICATE ROWS PRESENT.
**Note** Having duplicate rows in a dataset skews data analysis and distorts statistical metrics

## 1.3) Data Inquiry

In [5]:
print(df.duplicated().sum())

0


#### Perfect! NO Duplicate Rows Present

### 1.4) Structural Inspection

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_approved_at              99281 non-null  str  
 5   order_delivered_carrier_date   97658 non-null  str  
 6   order_delivered_customer_date  96476 non-null  str  
 7   order_estimated_delivery_date  99441 non-null  str  
dtypes: str(8)
memory usage: 21.9 MB


Since these columns (**order_purchase_timestamp**, **order_approved_at**, 
**order_delivered_carrier_date**, **order_delivered_customer_date**, 
**order_estimated_delivery_date**) have to be timestamps but here they are strings (str), 
we have to convert them to timestamps.

In [7]:
# I made a List of all datetime columns that are currently strings to convert them to DateTime
datetime_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

# Using FOR Loop I converted those columns to DateTime DataType
for col in datetime_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')

# TO verfiy Changes
print(df[datetime_cols].dtypes)

order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object


In [8]:
# I ran this again to check does the Datatypes converted successfully
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  str           
 1   customer_id                    99441 non-null  str           
 2   order_status                   99441 non-null  str           
 3   order_purchase_timestamp       99441 non-null  datetime64[us]
 4   order_approved_at              99281 non-null  datetime64[us]
 5   order_delivered_carrier_date   97658 non-null  datetime64[us]
 6   order_delivered_customer_date  96476 non-null  datetime64[us]
 7   order_estimated_delivery_date  99441 non-null  datetime64[us]
dtypes: datetime64[us](5), str(3)
memory usage: 13.0 MB


DONE!
## 1.4) Missing Data Audit

In [9]:
df.isnull().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

To fill the 160 null values in the **order_approved_at** column accurately,\
I checked the **order_status** for these specific records.\
Analyzing this distribution allows me to apply a targeted data-cleaning strategy,\
as I can address the **missing approval timestamps** based on whether the order was canceled, created, or delivered.

In [10]:
# 1. Applied Filter for the rows where approval time is missing
missing_approval = df[df['order_approved_at'].isnull()]

# 2. Get the exact count for all labels in order_status
status_counts = missing_approval['order_status'].value_counts()

print(status_counts)

order_status
canceled     141
delivered     14
created        5
Name: count, dtype: int64


Out of the **160** missing values, 141 belong to **canceled orders**, which naturally do not require an approval timestamp.\
Additionally, the 5 created orders can be left unfilled as their count is negligible.\
To handle the remaining 14 delivered orders accurately,\
I will calculate the historical time gap between **order_purchase_timestamp** and **order_approved_at** across the rest of the dataset.\
I will then compute the average duration of this approval gap and use it to impute the missing **order_approved_at** values for these 14 records.


In [11]:
# 1. Calculate the actual time gap for all successfully approved orders
valid_approvals = df['order_approved_at'].notnull()

time_gaps = (
    df.loc[valid_approvals, 'order_approved_at']
    - df.loc[valid_approvals, 'order_purchase_timestamp']
)

# 2. Calculate the average approval delay
avg_approval_delay = time_gaps.mean().round('s')

# 3. Identify the 14 delivered orders with missing approval timestamps
target_rows_mask = (
    df['order_approved_at'].isnull()
    & (df['order_status'] == 'delivered')
)

# 4. Display 5 sample rows BEFORE filling
print("=" * 80)
print("STEP 1: SAMPLE OF 5 ROWS BEFORE FILLING")
print("=" * 80)

print(
    df.loc[
        target_rows_mask,
        ['order_id', 'order_purchase_timestamp',
         'order_approved_at', 'order_status']
    ].head(5).to_string(index=False)
)

# 5. Display the calculated average approval delay
print("\n" + "=" * 80)
print("STEP 2: CALCULATED AVERAGE APPROVAL DELAY")
print("=" * 80)

print(f"Average approval delay: {avg_approval_delay}")
print(f"Average approval delay in hours: {avg_approval_delay.total_seconds() / 3600:.2f} hours")

# 6. Fill the missing approval timestamps for the 14 target rows
df.loc[target_rows_mask, 'order_approved_at'] = (
    df.loc[target_rows_mask, 'order_purchase_timestamp']
    + avg_approval_delay
)

# 7. Display ALL 14 rows AFTER filling
print("\n" + "=" * 80)
print("STEP 3: ALL TARGET ROWS AFTER FILLING")
print("=" * 80)

print(
    df.loc[
        target_rows_mask,
        ['order_id', 'order_purchase_timestamp',
         'order_approved_at', 'order_status']
    ].to_string(index=False)
)


STEP 1: SAMPLE OF 5 ROWS BEFORE FILLING
                        order_id order_purchase_timestamp order_approved_at order_status
e04abd8149ef81b95221e88f6ed9ab6a      2017-02-18 14:40:00               NaT    delivered
8a9adc69528e1001fc68dd0aaebbb54a      2017-02-18 12:45:31               NaT    delivered
7013bcfc1c97fe719a7b5e05e61c12db      2017-02-18 13:29:47               NaT    delivered
5cf925b116421afa85ee25e99b4c34fb      2017-02-18 16:48:35               NaT    delivered
12a95a3c06dbaec84bcfb0e2da5d228a      2017-02-17 13:05:55               NaT    delivered

STEP 2: CALCULATED AVERAGE APPROVAL DELAY
Average approval delay: 0 days 10:25:09
Average approval delay in hours: 10.42 hours

STEP 3: ALL TARGET ROWS AFTER FILLING
                        order_id order_purchase_timestamp   order_approved_at order_status
e04abd8149ef81b95221e88f6ed9ab6a      2017-02-18 14:40:00 2017-02-19 01:05:09    delivered
8a9adc69528e1001fc68dd0aaebbb54a      2017-02-18 12:45:31 2017-02-18 23:10:40

In [12]:
# 1. we have to apply Filter for the rows where approval time is missing
missing_approval = df[df['order_approved_at'].isnull()]

# 2. NOW we Have to Get the exact count for all labels in order_status
status_counts = missing_approval['order_status'].value_counts()

print(status_counts)

order_status
canceled    141
created       5
Name: count, dtype: int64


I found 1,783 missing values in **order_delivered_carrier_date**, which records when the order was handed over to the carrier.\

I decided to preserve these missing values rather than fill them with an average timestamp.\
**My reasons**\
**1.** The carrier handoff may never have happened. Canceled and unavailable orders may never have reached the shipping stage.\
**2.** I should not create events that are not supported by the data. Filling missing carrier dates with estimated timestamps could make it appear that orders were handed over to carriers when this may not have occurred.\
**3.** Imputation could distort shipping metrics. Artificial timestamps could affect average shipping preparation times, transit times, and delivery performance calculations.\
**4.** I can still analyze the available data. I can calculate shipping and delivery metrics using orders with valid timestamps, while documenting the number of records excluded from each metric.\
**5.** Missing values are not always data errors. Some timestamps are missing because the order may not have reached the corresponding stage of the fulfillment process.

In [13]:
# 1. Count missing carrier dates by order status
missing_carrier_dates = df[
    df['order_delivered_carrier_date'].isnull()
]

print("Missing carrier dates by order status:")
print(missing_carrier_dates['order_status'].value_counts(dropna=False))

print("\nTotal missing carrier dates:", len(missing_carrier_dates))

Missing carrier dates by order status:
order_status
unavailable    609
canceled       550
invoiced       314
processing     301
created          5
approved         2
delivered        2
Name: count, dtype: int64

Total missing carrier dates: 1783


In [14]:
# Count missing customer delivery dates by order status
missing_delivery = df[
    df['order_delivered_customer_date'].isnull()
]

print("Missing delivery dates by order status:")
print(missing_delivery['order_status'].value_counts(dropna=False))

print("\nTotal missing delivery dates:", len(missing_delivery))

Missing delivery dates by order status:
order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64

Total missing delivery dates: 2965


### Reasons for Leaving Missing Customer Delivery Dates as Null

- The actual delivery dates are not available in the dataset.
- Estimated delivery dates do not represent actual delivery dates.
- Filling missing values with averages could produce inaccurate delivery times.
- Incorrect timestamps could affect delivery performance analysis and business decisions.
- Only valid timestamps will be used to calculate delivery metrics.
- The missing values will be retained to maintain data accuracy and integrity.

In [15]:
delivered_missing_date = df[
    df['order_status'].eq('delivered')
    & df['order_delivered_customer_date'].isnull()
]

print(
    delivered_missing_date[
        [
            'order_id',
            'order_purchase_timestamp',
            'order_approved_at',
            'order_delivered_carrier_date',
            'order_delivered_customer_date',
            'order_estimated_delivery_date',
            'order_status'
        ]
    ].to_string(index=False)
)

                        order_id order_purchase_timestamp   order_approved_at order_delivered_carrier_date order_delivered_customer_date order_estimated_delivery_date order_status
2d1e2d5bf4dc7227b3bfebb81328c15f      2017-11-28 17:44:07 2017-11-28 17:56:40          2017-11-30 18:12:23                           NaT                    2017-12-18    delivered
f5dd62b788049ad9fc0526e3ad11a097      2018-06-20 06:58:43 2018-06-20 07:19:05          2018-06-25 08:05:00                           NaT                    2018-07-16    delivered
2ebdfc4f15f23b91474edf87475f108e      2018-07-01 17:05:11 2018-07-01 17:15:12          2018-07-03 13:57:00                           NaT                    2018-07-30    delivered
e69f75a717d64fc5ecdfae42b2e8e086      2018-07-01 22:05:55 2018-07-01 22:15:14          2018-07-03 13:57:00                           NaT                    2018-07-30    delivered
0d3268bad9b086af767785e3f0fc0133      2018-07-01 21:14:02 2018-07-01 21:29:54          2018-07-03 09

In [16]:
pip install sqlalchemy "psycopg[binary]" python-dotenv jupyter

  Using cached fqdn-1.6.0-py3-none-any.whl.metadata (9.4 kB)
  Using cached isoduration-20.11.0-py3-none-any.whl.metadata (5.7 kB)
  Using cached rfc3987_syntax-1.1.0-py3-none-any.whl.metadata (7.7 kB)
  Using cached uri_template-1.3.0-py3-none-any.whl.metadata (8.8 kB)
  Using cached webcolors-25.10.0-py3-none-any.whl.metadata (2.2 kB)
  Using cached lark-1.3.1-py3-none-any.whl.metadata (1.8 kB)
   ---------------------------------------- 0.0/3.8 MB ? eta -:--:--
   ----------- ---------------------------- 1.0/3.8 MB 5.7 MB/s eta 0:00:01
   ------------------------- -------------- 2.4/3.8 MB 6.1 MB/s eta 0:00:01
   ---------------------------------------  3.7/3.8 MB 6.6 MB/s eta 0:00:01
   ---------------------------------------- 3.8/3.8 MB 5.8 MB/s  0:00:00
Using cached rfc3987_syntax-1.1.0-py3-none-any.whl (8.0 kB)
Using cached lark-1.3.1-py3-none-any.whl (113 kB)
Using cached webcolors-25.10.0-py3-none-any.whl (14 kB)
Using cached fqdn-1.6.0-py3-none-any.whl (12 kB)
Using cached is

In [21]:
import os
import pandas as pd
import psycopg

from pathlib import Path
from dotenv import load_dotenv
from sqlalchemy import URL, create_engine, text

# My credentials you should use Yours
PROJECT_ROOT = Path(r"C:\Users\abdul\Downloads\olist_ecommerce_analysis")
load_dotenv(PROJECT_ROOT / ".env", override=True)

host = os.environ["RETAIL_POSTGRES_HOST"]
port = int(os.environ["RETAIL_POSTGRES_PORT"])
dbname = os.environ["RETAIL_POSTGRES_DB"]
user = os.environ["RETAIL_POSTGRES_USER"]
password = os.environ["RETAIL_POSTGRES_PASSWORD"]

# 1. psycopg connection
with psycopg.connect(
    host=host,
    port=port,
    dbname=dbname,
    user=user,
    password=password,
    connect_timeout=5,
) as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT current_database(), current_user;")
        print("psycopg connected:", cur.fetchone())

# 2. Let's Create SQLAlchemy engine
url = URL.create(
    "postgresql+psycopg",
    username=user,
    password=password,
    host=host,
    port=port,
    database=dbname,
)

engine = create_engine(url, pool_pre_ping=True)

# 3. It Confirms connection
with engine.connect() as connection:
    print("SQLAlchemy connected:", connection.execute(
        text("SELECT current_database()")
    ).scalar_one())

# 4. Create schema
with engine.begin() as connection:
    connection.execute(
        text("CREATE SCHEMA IF NOT EXISTS raw")
    )

# 5. Load cleaned DataFrame into PostgreSQL
orders_df.to_sql(
    name="olist_orders",
    con=engine,
    schema="raw",
    if_exists="fail",
    index=False,
    chunksize=2000,
    method="multi",
)

print("Table created and data loaded successfully!")

psycopg connected: ('olist_ecommerce_analysis', 'postgres')
SQLAlchemy connected: olist_ecommerce_analysis


NameError: name 'orders_df' is not defined

In [24]:
orders_df.to_sql(
    name="olist_orders",
    con=engine,
    schema="raw",
    if_exists="fail",
    index=False,
    chunksize=2000,
    method="multi",
)

print("Cleaned orders loaded into PostgreSQL!")

Cleaned orders loaded into PostgreSQL!


In [25]:
from sqlalchemy import text

with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT
            COUNT(*) AS total_rows,
            COUNT(DISTINCT order_id) AS unique_orders
        FROM raw.olist_orders;
    """))

    print(result.fetchone())

(99441, 99441)
